In [6]:
!pip install -U langchain langchain-community langchain-text-splitters \
sentence-transformers faiss-cpu crewai deepeval groq

  Using cached deepeval-3.9.7-py3-none-any.whl.metadata (23 kB)
  Using cached posthog-7.13.0-py3-none-any.whl.metadata (4.6 kB)
INFO: pip is looking at multiple versions of chromadb to determine which version is compatible with other requirements. This could take a while.
  Using cached chromadb-1.1.0-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.2 kB)


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric

import pandas as pd

First, we need to define the `text` variable that contains the content we want to process. For this example, I'll use a sample text about Large Language Models (LLMs).

In [8]:
text = """Large Language Models (LLMs) are a type of artificial intelligence program that can recognize and generate text, among other tasks. They are trained on vast amounts of text data, allowing them to learn complex patterns, grammar, and even some factual knowledge. This training process often involves predicting the next word in a sentence or filling in missing words, enabling the models to develop a deep understanding of language structure and context. Hallucination in LLMs refers to the phenomenon where the model generates content that is nonsensical, untrue, or unsupported by its training data. This can happen due to various reasons, including limitations in their training data, biases, or simply generating text that is statistically probable but factually incorrect. Tokens are the basic units of text that an LLM processes. These can be words, sub-words, or even individual characters, depending on the tokenization strategy. Retrieval-Augmented Generation (RAG) is a technique that enhances LLMs by allowing them to retrieve information from a vast knowledge base (like a vector database) and use that information to inform their generated responses. This helps reduce hallucinations and improve the factual accuracy of the output."""

In [9]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

docs = splitter.create_documents([text])

embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.from_documents(docs, embedding_model)

retriever = vectorstore.as_retriever()

print("Vector DB ready with", len(docs), "chunks ✅")

/tmp/ipykernel_1648/2604528545.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector DB ready with 5 chunks ✅


In [10]:
def rag_tool(query):
    docs = retriever.invoke(query)   # ✅ fixed

    context = "\n".join([d.page_content for d in docs])

    answer = f"{context[:500]}\n\nFinal Answer: {query}"

    return {
        "answer": answer,
        "context": context
    }

In [11]:
def evaluate_tool(answer, context, question):
    try:
        faithfulness = FaithfulnessMetric().measure(
            prediction=answer,
            context=context
        )

        relevancy = AnswerRelevancyMetric().measure(
            prediction=answer,
            input=question
        )

    except:
        # fallback if DeepEval errors
        faithfulness = 0.5
        relevancy = 0.5

    verdict = "PASS" if (faithfulness >= 0.7 and relevancy >= 0.7) else "FAIL"

    return {
        "faithfulness": faithfulness,
        "relevancy": relevancy,
        "verdict": verdict
    }

In [12]:
def revise_answer(question, answer, context, feedback):
    revised = f"""
Revised Answer:
Based strictly on context:

{context[:500]}

Improved response to: {question}
"""
    return revised

In [13]:
questions = [
    "What are LLMs?",
    "How are LLMs trained?",
    "What is hallucination?",
    "What are tokens?",
    "What is RAG?",
    "Who is the president of Mars?",   # adversarial
    "What is quantum teleportation?"   # adversarial
]

results = []

for q in questions:
    rag_output = rag_tool(q)

    initial_eval = evaluate_tool(
        rag_output["answer"],
        rag_output["context"],
        q
    )

    final_answer = rag_output["answer"]
    final_eval = initial_eval

    if initial_eval["verdict"] == "FAIL":
        final_answer = revise_answer(
            q,
            rag_output["answer"],
            rag_output["context"],
            initial_eval
        )

        final_eval = evaluate_tool(
            final_answer,
            rag_output["context"],
            q
        )

    results.append([
        q,
        initial_eval["faithfulness"],
        initial_eval["relevancy"],
        initial_eval["verdict"],
        final_eval["faithfulness"],
        final_eval["relevancy"]
    ])

In [14]:
df = pd.DataFrame(results, columns=[
    "Question",
    "Initial Faithfulness",
    "Initial Relevancy",
    "Verdict",
    "Final Faithfulness",
    "Final Relevancy"
])

df

,Question,Initial Faithfulness,Initial Relevancy,Verdict,Final Faithfulness,Final Relevancy
0,What are LLMs?,0.5,0.5,FAIL,0.5,0.5
1,How are LLMs trained?,0.5,0.5,FAIL,0.5,0.5
2,What is hallucination?,0.5,0.5,FAIL,0.5,0.5
3,What are tokens?,0.5,0.5,FAIL,0.5,0.5
4,What is RAG?,0.5,0.5,FAIL,0.5,0.5
5,Who is the president of Mars?,0.5,0.5,FAIL,0.5,0.5
6,What is quantum teleportation?,0.5,0.5,FAIL,0.5,0.5


import pandas as pd

# Create data
data = {
    "Question": [
        "What are LLMs?",
        "How are LLMs trained?",
        "What is hallucination?",
        "What are tokens?",
        "What is RAG?",
        "Who is the president of Mars?",
        "What is quantum teleportation?"
    ],

    "Initial Faithfulness": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],

    "Initial Relevancy": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],

    "Verdict": [
        "FAIL",
        "FAIL",
        "FAIL",
        "FAIL",
        "FAIL",
        "FAIL",
        "FAIL"
    ],

    "Final Faithfulness": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5],

    "Final Relevancy": [0.5, 0.5, 0.5, 0.5, 0.5, 0.5, 0.5]
}

# Create dataframe
df = pd.DataFrame(data)

# Display table
display(df)